In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# from dask.distributed import Client, LocalCluster
from genpp.configs import register_resolvers

register_resolvers()

In [3]:
import hydra
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

from genpp import BASE_DIR

CONFIG_DIR = BASE_DIR / "configs"
with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="base_fmunet")
    OmegaConf.resolve(cfg)

In [4]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()

Configuration hash: 05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c
Cached tensor data found. Verifying configuration...
Using cached tensor data.


In [5]:
datamodule.setup("fit")
train_dl = datamodule.train_dataloader()

In [6]:
for batch in train_dl:
    break

/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [7]:
import torch

model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)
# for testing purposes
model.register_buffer("scale_variance_td", torch.ones(2, 2))

In [8]:
model.n_samples = 3  # for testing purposes
model.step_size = 0.5  # for testing purposes
res = model.predict_step(batch)  # forward pass

works


In [9]:
res.shape

torch.Size([64, 3, 2, 37, 31])